In [1]:
import cv2
import mediapipe as mp
from scipy.spatial import distance as dist
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

def eye_aspect_ratio(eye_points):
    A = dist.euclidean(eye_points[1], eye_points[5])
    B = dist.euclidean(eye_points[2], eye_points[4])
    C = dist.euclidean(eye_points[0], eye_points[3])
    return (A + B) / (2.0 * C)

cap = cv2.VideoCapture(0)
EAR_THRESHOLD = 0.22
CONSEC_FRAMES = 3

# Use a dictionary to track counters per face
face_counters = {}

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)

        if results.multi_face_landmarks:
            # Loop through each detected face
            for idx, face_landmarks in enumerate(results.multi_face_landmarks):
                # Right eye
                right_eye = [(face_landmarks.landmark[i].x * frame.shape[1],
                              face_landmarks.landmark[i].y * frame.shape[0])
                             for i in [33, 160, 158, 133, 153, 144]]

                # Left eye
                left_eye = [(face_landmarks.landmark[i].x * frame.shape[1],
                             face_landmarks.landmark[i].y * frame.shape[0])
                            for i in [263, 387, 385, 362, 380, 373]]

                leftEAR = eye_aspect_ratio(left_eye)
                rightEAR = eye_aspect_ratio(right_eye)
                ear = (leftEAR + rightEAR) / 2.0

                # Initialize counter for new face
                if idx not in face_counters:
                    face_counters[idx] = 0

                # Update counter
                if ear < EAR_THRESHOLD:
                    face_counters[idx] += 1
                else:
                    face_counters[idx] = 0

                # Decide status
                if face_counters[idx] >= CONSEC_FRAMES:
                    status_text = f"User {idx+1}: LED ON (Eyes Closed)"
                    color = (0, 0, 255)
                else:
                    status_text = f"User {idx+1}: LED OFF (Eyes Open)"
                    color = (0, 255, 0)

                # Draw EAR and status on frame
                cv2.putText(frame, f"User {idx+1} EAR: {ear:.2f}",
                            (10, 30 + idx*50), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (0, 255, 0), 2)
                cv2.putText(frame, status_text,
                            (10, 55 + idx*50), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, color, 2)

        else:
            # Reset counters if no faces detected
            face_counters = {}

        cv2.imshow("Multi-User Blink Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()


C:\Users\Janani\anaconda3\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [2]:
import cv2
import mediapipe as mp
from scipy.spatial import distance as dist
import serial
import time

# Initialize Arduino connection (change 'COM3' to your Arduino port, e.g. '/dev/ttyUSB0' on Linux)
arduino = serial.Serial('COM7', 9600)
time.sleep(2)  # wait for Arduino reset

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

def eye_aspect_ratio(eye_points):
    A = dist.euclidean(eye_points[1], eye_points[5])
    B = dist.euclidean(eye_points[2], eye_points[4])
    C = dist.euclidean(eye_points[0], eye_points[3])
    return (A + B) / (2.0 * C)

cap = cv2.VideoCapture(0)  # use 0 for default webcam
EAR_THRESHOLD = 0.22
CONSEC_FRAMES = 3
face_counters = {}

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_frame)

        led_status = '0'  # default OFF

        if results.multi_face_landmarks:
            for idx, face_landmarks in enumerate(results.multi_face_landmarks):
                right_eye = [(face_landmarks.landmark[i].x * frame.shape[1],
                              face_landmarks.landmark[i].y * frame.shape[0])
                             for i in [33, 160, 158, 133, 153, 144]]
                left_eye = [(face_landmarks.landmark[i].x * frame.shape[1],
                             face_landmarks.landmark[i].y * frame.shape[0])
                            for i in [263, 387, 385, 362, 380, 373]]

                leftEAR = eye_aspect_ratio(left_eye)
                rightEAR = eye_aspect_ratio(right_eye)
                ear = (leftEAR + rightEAR) / 2.0

                if idx not in face_counters:
                    face_counters[idx] = 0

                if ear < EAR_THRESHOLD:
                    face_counters[idx] += 1
                else:
                    face_counters[idx] = 0

                if face_counters[idx] >= CONSEC_FRAMES:
                    status_text = f"User {idx+1}: LED ON (Eyes Closed)"
                    color = (0, 0, 255)
                    led_status = '1'  # ON
                else:
                    status_text = f"User {idx+1}: LED OFF (Eyes Open)"
                    color = (0, 255, 0)
                    led_status = '0'  # OFF

                cv2.putText(frame, f"User {idx+1} EAR: {ear:.2f}",
                            (10, 30 + idx*50), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (0, 255, 0), 2)
                cv2.putText(frame, status_text,
                            (10, 55 + idx*50), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, color, 2)

        else:
            face_counters = {}
            led_status = '0'

        # Send command to Arduino
        arduino.write(led_status.encode())
        
        cv2.imshow("Blink Detection with LED", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    arduino.close()

SerialException: could not open port 'COM7': FileNotFoundError(2, 'The system cannot find the file specified.', None, 2)